# MOB fixo x MOB evoluindo x Heatmap -- direto de `dias_atraso`, sem bucket

Notebook novo, do zero, pra separar os dois conceitos de MOB com clareza:

- **MOB fixo** -- corte transversal: fixa a IDADE, deixa a SAFRA variar com o
  calendário. Eixo X = `ref_month`. Responde "essa idade específica está
  piorando com o tempo?". É o mesmo conceito das curvas "OverXXMY" de um
  dashboard de referência.
- **MOB evoluindo** -- corte longitudinal: fixa a SAFRA, deixa a IDADE
  variar. Eixo X = `months_on_book`. Responde "essa safra específica, como
  ela amadurece?". É a curva de safra clássica.
- **Heatmap** -- as duas juntas, na mesma matriz (`ref_month` × MOB), sem
  escolher um corte só -- cada célula é a taxa de inadimplência pro limiar
  de atraso selecionado no slider.

**As três vêm de UMA tabela só** (`preparar_taxa_over_spark`) -- não tem
bucket em nenhum lugar, tudo calculado direto de `dias_atraso` para vários
limiares de "over" ao mesmo tempo (over15, over20, over30, ...).

**Pré-requisito:** `df_painel_confiavel` já na sessão. `%pip install plotly`
antes, se o cluster não tiver.


In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import plotly.graph_objects as go


## Preparação -- uma função só, para as três visões

Agrupa por `(ref_month, months_on_book)` -- `safra` entra como coluna
(é determinada pelas outras duas, `F.first` só recupera o valor). Para
cada limiar em `limiares`, calcula `taxa_overXX` numa única passada pela
base (não uma agregação por limiar).


In [ ]:
def preparar_taxa_over_spark(df_painel, limiares=(15, 20, 30, 45, 60, 90, 120, 180), mob_maximo: int = 24) -> pd.DataFrame:
    """
    Tabela única (ref_month, months_on_book) com taxa de atraso >= cada
    limiar em `limiares`, direto de `dias_atraso` -- SEM usar atraso_bucket
    em nenhum ponto. `safra` entra como coluna (F.first -- e determinada
    por ref_month e months_on_book, não precisa de groupBy por ela também).
    Serve as 3 visualizacoes: MOB fixo, MOB evoluindo, e o heatmap.
    """
    filtrado = df_painel.filter(F.col("months_on_book") <= mob_maximo)
    agregacoes = [F.avg((F.col("dias_atraso") >= lim).cast("double")).alias(f"taxa_over{lim}") for lim in limiares]
    agregacoes += [F.first("safra").alias("safra"), F.count(F.lit(1)).alias("n_contratos")]
    resultado = (
        filtrado.groupBy("ref_month", "months_on_book").agg(*agregacoes)
        .orderBy("ref_month", "months_on_book")
    )
    return resultado.toPandas()


## Visão 1 -- MOB fixo (eixo X = `ref_month`)

Uma linha por MOB escolhido -- cada ponto de uma linha é uma safra
diferente (mesma idade, calendário andando).


In [ ]:
def plot_mob_fixo(tabela: pd.DataFrame, limiar: int = 30, mobs_linha=(3, 6, 12)) -> go.Figure:
    coluna = f"taxa_over{limiar}"
    fig = go.Figure()
    for mob in mobs_linha:
        sub = tabela[tabela["months_on_book"] == mob].sort_values("ref_month")
        fig.add_trace(go.Scatter(x=sub["ref_month"], y=sub[coluna], mode="lines+markers", name=f"MOB {mob}"))
    fig.update_layout(
        title=f"MOB fixo -- Over{limiar}, por mês de referência",
        xaxis_title="Mês de referência (ref_month)", yaxis_title=f"% em atraso >= {limiar} dias", height=420,
    )
    return fig


## Visão 2 -- MOB evoluindo (eixo X = `months_on_book`)

Uma linha por safra escolhida -- cada ponto de uma linha é um MOB
diferente da MESMA safra, amadurecendo. Se `safras_linha` não for
informado, escolhe automaticamente algumas safras espaçadas ao longo do
histórico (mesma lógica de `safras_destacadas` que já usamos nas curvas
de safra do notebook principal).


In [ ]:
def plot_mob_evoluindo(tabela: pd.DataFrame, limiar: int = 30, safras_linha=None, n_safras_auto: int = 5) -> go.Figure:
    coluna = f"taxa_over{limiar}"
    if safras_linha is None:
        todas_safras = sorted(tabela["safra"].unique())
        passo = max(1, len(todas_safras) // n_safras_auto)
        safras_linha = todas_safras[::passo]

    fig = go.Figure()
    for safra in safras_linha:
        sub = tabela[tabela["safra"] == safra].sort_values("months_on_book")
        rotulo = pd.Timestamp(safra).strftime("%Y-%m")
        fig.add_trace(go.Scatter(x=sub["months_on_book"], y=sub[coluna], mode="lines+markers", name=f"Safra {rotulo}"))
    fig.update_layout(
        title=f"MOB evoluindo -- Over{limiar}, por safra",
        xaxis_title="Months on Book", yaxis_title=f"% em atraso >= {limiar} dias", height=420,
    )
    return fig


## Visão 3 -- Heatmap (`ref_month` × MOB), com slider de limiar

As duas visões anteriores juntas, na matriz inteira. Cor = taxa, no
limiar selecionado no slider. Células com menos de `n_minimo` contratos
ficam cinza (camada cinza fixa por baixo + camada colorida com `NaN` nas
células de pouco dado).


In [ ]:
def plot_heatmap_over(tabela: pd.DataFrame, limiares=(15, 20, 30, 45, 60, 90, 120, 180), n_minimo: int = 200) -> go.Figure:
    todos_ref_month = sorted(tabela["ref_month"].unique())
    todos_mob = sorted(tabela["months_on_book"].unique())
    ref_month_str = [pd.Timestamp(d).strftime("%Y-%m") for d in todos_ref_month]

    def _matriz(lim):
        piv_taxa = tabela.pivot(index="ref_month", columns="months_on_book", values=f"taxa_over{lim}").reindex(index=todos_ref_month, columns=todos_mob)
        piv_n = tabela.pivot(index="ref_month", columns="months_on_book", values="n_contratos").reindex(index=todos_ref_month, columns=todos_mob)
        z = piv_taxa.values.copy()
        z[(piv_n.values < n_minimo) | pd.isna(piv_n.values)] = float("nan")
        return z

    z_fundo = [[0] * len(todos_mob) for _ in todos_ref_month]

    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=z_fundo, x=todos_mob, y=ref_month_str,
                               colorscale=[[0, "#D9D9D9"], [1, "#D9D9D9"]], showscale=False, hoverinfo="skip"))
    fig.add_trace(go.Heatmap(z=_matriz(limiares[len(limiares) // 2]), x=todos_mob, y=ref_month_str,
                               colorscale="Reds", colorbar=dict(title="Taxa", tickformat=".0%"),
                               hovertemplate="Foto: %{y}<br>MOB: %{x}<br>Taxa: %{z:.2%}<extra></extra>"))

    fig.frames = [go.Frame(data=[go.Heatmap(z=z_fundo), go.Heatmap(z=_matriz(lim))], name=str(lim)) for lim in limiares]
    fig.update_layout(
        title="Taxa de inadimplência por ref_month × MOB, por limiar de atraso (cinza = poucos contratos)",
        xaxis_title="Months on Book", yaxis_title="Mês de referência (foto)", height=650,
        sliders=[{
            "steps": [{"args": [[str(lim)], {"frame": {"duration": 0}, "mode": "immediate"}],
                        "label": f"Over{lim}", "method": "animate"} for lim in limiares],
            "currentvalue": {"prefix": "Limiar: "},
        }],
    )
    return fig


## Como usar

Uma preparação só, alimentando as três visões:


In [ ]:
tabela = preparar_taxa_over_spark(df_painel_confiavel, limiares=(15, 20, 30, 45, 60, 90, 120, 180))

fig_mob_fixo = plot_mob_fixo(tabela, limiar=30, mobs_linha=(3, 6, 12))
fig_mob_fixo.show()

fig_mob_evoluindo = plot_mob_evoluindo(tabela, limiar=30)
fig_mob_evoluindo.show()

fig_heatmap = plot_heatmap_over(tabela, n_minimo=200)
fig_heatmap.show()


---
**Nota de transparência:** não executado contra Spark real nesta sessão
(sem PySpark disponível aqui) -- a lógica de agregação segue o mesmo
padrão já validado no resto do pipeline. A lógica de plot (as 3 funções)
foi testada com dado fabricado no formato que `.toPandas()` devolveria,
antes desta entrega.
